# Generating QRC inputs from Gaussian 16 frequency calculations

pyQRC reads a completed frequency calculation and writes a new input file whose geometry has been displaced along one or more normal modes — Silva and Goodman's *Quick Reaction Coordinate* (QRC) approach. This notebook walks through the Gaussian 16 example files that ship in this directory.

Requirements: `pip install pyqrc` (pulls in cclib and numpy).


## Setup

pyQRC writes its new input files next to the file it reads, so we copy the example outputs into a `scratch/` subdirectory and run everything there. The helper below invokes the same `pyqrc` command line you would use in a terminal or HPC batch script.

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

HERE = Path.cwd()                # this examples directory
SCRATCH = HERE / "scratch"
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)
SCRATCH.mkdir()

for name in ['acetaldehyde.log', 'claisen_ts.log', 'planar_chex.log']:
    shutil.copy(HERE / name, SCRATCH / name)


def run_pyqrc(*args):
    """Run the pyqrc command line inside the scratch directory."""
    result = subprocess.run(
        [sys.executable, "-m", "pyqrc", *args],
        cwd=SCRATCH, capture_output=True, text=True,
    )
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr, end="")
        raise RuntimeError(f"pyqrc exited with code {result.returncode}")


def show(filename, n=40):
    """Print up to n lines of a file in the scratch directory."""
    lines = (SCRATCH / filename).read_text().splitlines()
    print("\n".join(lines[:n]))
    if len(lines) > n:
        print(f"... ({len(lines) - n} more lines)")


## Example 1: remove an unwanted imaginary frequency

This acetaldehyde optimization inadvertently produced a saddle point — it has one small imaginary frequency. By default pyQRC displaces along **all** imaginary modes, which is exactly what we want here: the displaced geometry breaks the symmetry of the saddle point, and re-optimizing it gives the true minimum.

In [2]:
run_pyqrc("acetaldehyde.log", "--nproc", "4", "--mem", "8GB")

o   acetaldehyde.log had 1 imaginary frequencies: processed


That wrote two files: `acetaldehyde_QRC.com` (the new, displaced input — ready to submit) and `acetaldehyde_QRC.qrc` (a human-readable summary of the frequencies and the displacement).

In [3]:
show("acetaldehyde_QRC.com")

%chk=acetaldehyde_QRC.chk
%nproc=4
%mem=8GB
# opt freq M062X/6-31G*

acetaldehyde_QRC

0 1
 C  -0.24032600   0.41382900  -0.01800100
 O  -1.21931300  -0.28867900   0.01400000
 H  -0.34388700   1.51866800  -0.08199900
 C   1.16822300  -0.13555600   0.00399900
 H   1.91748100   0.65773200   0.10393500
 H   1.26478600  -0.85024500   0.83156500
 H   1.34874500  -0.68635700  -0.93149200



In [4]:
show("acetaldehyde_QRC.qrc", n=24)

 pyQRC - a quick alternative to IRC calculations
 version: 2.3.0 / author: Robert Paton / email: robert.paton@colostate.edu
 Based on: Goodman, J. M.; Silva, M. A. Tet. Lett. 2003, 44, 8233-8236;
 Tet. Lett. 2005, 46, 2067-2069.

                -----ORIGINAL GEOMETRY------
                       X         Y         Z
   C           -0.240326  0.413829 -0.000001
   O           -1.219313 -0.288679  0.000000
   H           -0.343887  1.518668  0.000001
   C            1.168223 -0.135556 -0.000001
   H            1.917481  0.657732 -0.000065
   H            1.306786 -0.768245  0.881565
   H            1.306745 -0.768357 -0.881492

                ----HARMONIC FREQUENCIES----
                    Freq  Red mass   F const
               -164.6818    1.1913    0.0190
                507.9671    2.6617    0.4046
                761.1498    1.1728    0.4003
                952.1134    2.1851    1.1671
               1115.6721    1.9956    1.4635
               1147.0025    1.7207    1.3338
    

## Example 2: map a reaction coordinate (the namesake QRC)

For a transition state — here a Claisen rearrangement — the quick alternative to an IRC is two displaced inputs: one along the imaginary mode (`--amp 0.3`) and one in the reverse direction (`--amp -0.3`). Optimizing both gives the reactant and product the TS connects. The benchmark in the README found an amplitude of **0.3** performs best, hence the values used here; `--name` controls the suffix of the generated files.

In [5]:
run_pyqrc("claisen_ts.log", "--nproc", "4", "--mem", "8GB", "--amp", "0.3", "--name", "QRCF")
run_pyqrc("claisen_ts.log", "--nproc", "4", "--mem", "8GB", "--amp", "-0.3", "--name", "QRCR")

o   claisen_ts.log had 1 imaginary frequencies: processed
o   claisen_ts.log had 1 imaginary frequencies: processed


In [6]:
show("claisen_ts_QRCF.com", n=14)
print("=" * 60)
show("claisen_ts_QRCR.com", n=14)

%chk=claisen_ts_QRCF.chk
%nproc=4
%mem=8GB
# opt(ts,calcfc,noeigentest) freq=noraman wb97xd/6-31+G*

claisen_ts_QRCF

0 1
 C  -1.37087800   0.76347400  -0.26922500
 C  -1.24135200  -0.56957300   0.25254000
 O  -0.51557700  -1.40777800  -0.27025500
 C   1.45620400  -0.77049900   0.20597600
 C   1.33983300   0.47219400  -0.29750300
 C   0.42652100   1.41099800   0.30589200
... (9 more lines)
%chk=claisen_ts_QRCR.chk
%nproc=4
%mem=8GB
# opt(ts,calcfc,noeigentest) freq=noraman wb97xd/6-31+G*

claisen_ts_QRCR

0 1
 C  -1.57487800   0.64347400  -0.31722500
 C  -1.24735200  -0.51557300   0.25854000
 O  -0.29957700  -1.34777800  -0.23425500
 C   1.17420400  -0.87849900   0.15797600
 C   1.32183300   0.51419400  -0.29750300
 C   0.66052100   1.47699800   0.34789200
... (9 more lines)


## Example 3: conformational sampling via specific modes

Planar cyclohexane is a third-order saddle point with three imaginary frequencies. Instead of displacing along all of them at once, `--freqnum N` picks a single (1-indexed, sorted from lowest) mode. Optimizing the two inputs below yields different minima: mode 1 leads to the chair, mode 3 to the twist-boat — a simple form of conformational sampling. (`--freq <value>` similarly picks the mode nearest a wavenumber in cm⁻¹.)

In [7]:
run_pyqrc("planar_chex.log", "--nproc", "4", "--freqnum", "1", "--name", "mode1")
run_pyqrc("planar_chex.log", "--nproc", "4", "--freqnum", "3", "--name", "mode3")

o   planar_chex.log was distorted along freq #1
o   planar_chex.log was distorted along freq #3


In [8]:
show("planar_chex_mode1.com", n=10)
print("=" * 60)
show("planar_chex_mode3.com", n=10)

%chk=planar_chex_mode1.chk
%nproc=4
%mem=4GB
# symm=loose opt b3lyp/6-31G* freq

planar_chex_mode1

0 1
 C   0.16785800   1.54628900   0.01792900
 C   1.42312600   0.62775200  -0.01724200
... (17 more lines)
%chk=planar_chex_mode3.chk
%nproc=4
%mem=4GB
# symm=loose opt b3lyp/6-31G* freq

planar_chex_mode3

0 1
 C   0.16785800   1.54628900   0.02592900
 C   1.42312600   0.62775200  -0.01924200
... (17 more lines)


## Using the Python API instead of the CLI

The same machinery is importable. `QRCGenerator` parses the output, computes the displaced geometry, and (unless `write=False`) writes the files in one go. With `write=False` you can inspect the displacement before committing anything to disk.

In [9]:
import numpy as np
from pyqrc import QRCGenerator

qrc = QRCGenerator(
    file=str(SCRATCH / "claisen_ts.log"),
    amplitude=0.3,
    nproc=4,
    mem="8GB",
    route=None,     # None clones the route/keywords from the original job
    verbose=False,  # skip the .qrc summary file
    suffix="API",
    val=None,       # or displace along the mode nearest this frequency (cm-1)
    num=None,       # or along this 1-indexed mode number
    write=False,    # compute only; no files are written
)

freqs = np.asarray(qrc.FREQS)
print("Imaginary frequencies (cm-1):", freqs[freqs < 0.0])
print(f"Mass-weighted displacement from the TS: {qrc.MW_DISTANCE:.4f} bohr amu^1/2")
print("Displaced geometry has clashing atoms:", qrc.OVERLAPPED)
print()

per_atom = np.linalg.norm(qrc.NEW_CARTESIAN - qrc.CARTESIAN, axis=1)
print("Atoms that move the most:")
for i in np.argsort(per_atom)[::-1][:5]:
    print(f"  atom {i + 1:>2} ({qrc.ATOMTYPES[i]:<2}) moved {per_atom[i]:.3f} Angstrom")


Imaginary frequencies (cm-1): [-589.7505]
Mass-weighted displacement from the TS: 1.7753 bohr amu^1/2
Displaced geometry has clashing atoms: False

Atoms that move the most:
  atom  4 (C ) moved 0.153 Angstrom
  atom  6 (C ) moved 0.123 Angstrom
  atom  1 (C ) moved 0.121 Angstrom
  atom  3 (O ) moved 0.114 Angstrom
  atom  7 (H ) moved 0.098 Angstrom


## Where the files went

Everything generated above is in the `scratch/` subdirectory (ignored by git) — delete it when you are done. In real use you would submit the new input file (`*.com`) to your scheduler and optimize.

See the [project README](../../README.md) for the full option list and for the IRC-comparison benchmark behind the recommended amplitude of 0.3.
